# Generate caravan Map
In this notebook you will retrieve the geospatial data for the Cravan dataset.The caravan dataset is a collection of streamflow and forcing data. <br>
Caravan was prepared by [Frederik Kratzert](https://doi.org/10.1038/s41597-023-01975-w), the forcing is based on the ERA5-Land model. The streamflow is from the USGS. <br>
To access it easily, it was stored [here](https://doi.org/10.4121/bf0eaf7c-f2fa-46f6-b8cd-77ad939dd350.v4) on the [OPenDAP](https://data.4tu.nl/info/about-your-data/netcdf-and-opendap) server from data.4TU.nl .<br>
This saves you from downloading and reading the whole dataset hosted on [zenodo](https://zenodo.org/records/6578598), instead only the necesarry data is downloaded. 

This notebook will show case how to turn this into an easily accessible map.  

This supplies you with the basin_id.
The shapefiles will be downloaded automatically and the catchment information downloaded from the server


In [1]:
import wget
from zipp import zipfile
from pathlib import Path
import geopandas as gpd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import pandas as pd
import csv
import json

import requests
from bs4 import BeautifulSoup
import numpy as np

import folium

Specify url

In [2]:
COMMON_URL = "ca13056c-c347-4a27-b320-930c2a4dd207"
OPENDAP_URL = f"https://opendap.4tu.nl/thredds/dodsC/data2/djht/{COMMON_URL}/1/"
SHAPEFILE_URL = (
    f"https://data.4tu.nl/file/{COMMON_URL}/bbe94526-cf1a-4b96-8155-244f20094719"
)
DIRECTORY = "/data/shared/climate-data/caravan/"

Download the required shapefiles from [data.4TU.nl](https://doi.org/10.4121/bf0eaf7c-f2fa-46f6-b8cd-77ad939dd350.v4)

In [3]:
out_path = Path('shapefiles')
out_path.mkdir(exist_ok=True)

# zip_path = out_path / 'shapefiles.zip'

# if not zip_path.is_file():
#     wget.download(SHAPEFILE_URL, out=str(zip_path))

In [4]:
# combined_shapefile_path = out_path / "combined.shp"
combined_shapefile_path = DIRECTORY + "shapefiles/combined.shp"
# if not combined_shapefile_path.is_file():
#     with zipfile.ZipFile(zip_path) as myzip:
#         myzip.extractall()

Load in the dataset

In [5]:
gdf = gpd.read_file(combined_shapefile_path)

In [6]:
dataset_names = list(set(gdf['gauge_id'].apply(lambda x: x.split('_')[0]).values))
dataset_names

['hysets', 'lamah', 'camelsgb', 'camelsaus', 'camelsbr', 'camels', 'camelscl']

In [7]:
map_path = Path('docs')
if not map_path.exists():
    map_path.mkdir()

In [8]:
# ax = plt.axes(projection=ccrs.PlateCarree())
# ax.add_feature(cfeature.COASTLINE, edgecolor='gray')
# gdf.plot(ax=ax,zorder=1,color="C0")
# plt.tight_layout()
# plt.savefig(map_path / 'overview_image.png',bbox_inches='tight',dpi=400)

## load in regions that are done already

In [9]:
REGION_STRUCTURE_FILE = Path("region_structure.json")
DONE_ROOT = Path("done")

def load_region_structure(path: Path = REGION_STRUCTURE_FILE) -> dict[str, list[str]]:
    """Load the {country: [camel_id, ...]} mapping of all possible regions."""
    with open(path, "r") as f:
        return json.load(f)


def load_done_regions(country: str, done_root: Path = DONE_ROOT) -> set[str]:
    """Read finished camel_ids from done/<country>/regions_done.csv."""
    csv_path = done_root / country / "regions_done.csv"
    done = set()
    if not csv_path.exists():
        return done

    with open(csv_path, "r", newline="") as f:
        reader = csv.reader(f)
        next(reader, None)  # skip header
        for row in reader:
            if row:
                done.add(row[0])
    return done


def get_todo_regions(country: str, structure: dict[str, list[str]]) -> list[str]:
    """Return camel_ids for a country that are NOT yet marked done."""
    all_ids = set(structure.get(country, []))
    done_ids = load_done_regions(country)
    return sorted(all_ids - done_ids)

structure = load_region_structure()

for c, ids in structure.items():
    done = load_done_regions(c)
    todo = set(ids) - done
    print(f"{c:<28} {len(ids):>6} {len(done):>6} {len(todo):>6}")

australia                       150     12    138
austria                         307     12    295
brazil                          376      2    374
canada                          886      3    883
chile                           314      1    313
czech_republic                   35      1     34
england                         247      2    245
germany                         120      1    119
lichtenstein                      1      1      0
mexico                           16      0     16
scotland                        124      2    122
switzerland                      16      6     10
united_states_of_america       4201      1   4200
wales                            37      5     32


In [10]:
# gdf["done"] = gdf["gauge_id"].isin(region_ids_done)

# # plt.figure(figsize=(21, 7)) 

# ax = plt.axes(projection=ccrs.PlateCarree())
# ax.add_feature(cfeature.COASTLINE, edgecolor='gray')

# # Plot all regions (default: blue)
# gdf[~gdf["done"]].plot(ax=ax, zorder=1, color="red")

# # Plot completed regions (green)
# gdf[gdf["done"]].plot(ax=ax, zorder=2, color="green")

# plt.tight_layout()
# plt.savefig(map_path / "overview_image_cci.png", bbox_inches='tight', dpi=1600)
# plt.show()

# Load data

In [11]:
import xarray as xr

In [12]:
def get_camels_ids(dataset: str):
    # ds = xr.open_dataset(f"{DIRECTORY}{dataset}.nc")
    ds = xr.open_dataset(f"{OPENDAP_URL}{dataset}.nc")
    return np.array([basin for basin in ds['basin_id'].values])

In [13]:
def get_caravan(dataset: str, basin_id: str):
    # ds = xr.open_dataset(f"{DIRECTORY}{dataset}.nc")
    ds = xr.open_dataset(f"{OPENDAP_URL}{dataset}.nc")
    return ds.sel(basin_id=basin_id)

In [14]:
def get_dataset(dataset) -> xr.Dataset:
    """Opens specified dataset from data.4tu.nl OPeNDAP server.
    Args:
        dataset (str): name of dataset, choose from:
            'camels',
            'camelsaus',
            'camelsbr',
            'camelscl',
            'camelsgb',
            'hysets',
            'lamah'
    """
    # return xr.open_dataset(f"{DIRECTORY}{dataset}.nc")
    return xr.open_dataset(f"{OPENDAP_URL}{dataset}.nc")


In [15]:
# get_camels_ids(dataset_names[-3])[0]

In [16]:
# basin_id= "camels_01022500"
# ds = get_caravan("camels", basin_id)
# print(ds)

In [17]:
PROPERTY_VARS = [
    "timezone",
    "name",
    "country",
    "lat",
    "lon",
    "area",
    "p_mean",
    "pet_mean",
    "aridity",
    "frac_snow",
    "moisture_index",
    "seasonality",
    "high_prec_freq",
    "high_prec_dur",
    "low_prec_freq",
    "low_prec_dur",
]
RENAME_ERA5 = {
    "total_precipitation_sum": "pr",
    "potential_evaporation_sum": "evspsblpot",
    "temperature_2m_mean": "tas",
    "temperature_2m_min": "tasmin",
    "temperature_2m_max": "tasmax",
    "streamflow": "Q",
}

In [18]:
# ds

In [19]:
# lst_df_info = []
# for dataset_name in dataset_names:
#     ds = get_dataset(dataset_name)
#     variables = ds.data_vars.keys()
#     properties = set(variables).intersection(PROPERTY_VARS)
#     non_property_vars = set(variables) - properties
#     variable_names = non_property_vars.intersection(
#         RENAME_ERA5.keys()
#     )  # only take the vars also in Rename dict
#     ds = ds.drop_vars(non_property_vars)
#     ds = ds.drop_indexes("time")
#     ds = ds.drop_vars("time")
#     ds['gauge_id'] = [val for val in ds['basin_id'].values]
    
#     df_info = ds.to_dataframe()
#     for val in ['timezone','name', 'country']:
#         # df_info[val] = df_info[val].apply(
#         #     lambda x: x.decode("ISO-8859-1") if isinstance(x, (bytes, bytearray)) else x
#         # )
#         df_info[val] = df_info[val]
    
#     lst_df_info.append(df_info)

In [20]:
lst_df_info = []
for dataset_name in dataset_names:
    ds = get_dataset(dataset_name)
    variables = ds.data_vars.keys()
    properties = set(variables).intersection(PROPERTY_VARS)
    non_property_vars = set(variables) - properties
    variable_names = non_property_vars.intersection(
        RENAME_ERA5.keys()
    )  # only take the vars also in Rename dict
    ds = ds.drop_vars(non_property_vars)
    ds = ds.drop_indexes("time")
    ds = ds.drop_vars("time")
    ds['basin_id'] = ds['basin_id'].astype(str).str.strip()

    df_info = ds.to_dataframe()
    for val in ['timezone','name', 'country']:
        df_info[val] = df_info[val].apply(
            lambda x: x.decode("ISO-8859-1") if isinstance(x, (bytes, bytearray)) else x
        )
        
    lst_df_info.append(df_info)

Error:curl error: Problem with the SSL CA cert (path? access rights?)
curl error details: 


OSError: [Errno -68] NetCDF: I/O failure: 'https://opendap.4tu.nl/thredds/dodsC/data2/djht/ca13056c-c347-4a27-b320-930c2a4dd207/1/hysets.nc'

In [ ]:
# print(lst_df_info)

In [ ]:
# df_combined_info = pd.concat(lst_df_info).sort_index()

normalized_dfs = []

for df in lst_df_info:
    # if MultiIndex, collapse to basin_id only
    if isinstance(df.index, pd.MultiIndex):
        df = df.reset_index()  # move all index levels to columns
        # keep only basin_id as index
        df = df.set_index('basin_id')
    else:
        df.index = df.index.astype(str).str.strip()  # ensure clean strings

    normalized_dfs.append(df)

# Now safe to concat and sort
df_combined_info = pd.concat(normalized_dfs).sort_index()
# print(df_combined_info)

In [23]:
# First, drop 'data_source' if it's causing issues
df_combined_info_clean = df_combined_info.drop(columns=['data_source'], errors='ignore')
# df_combined_info_clean = df_combined_info
# Sum numeric columns grouped by basin_id
df_aggregated = df_combined_info_clean.groupby('basin_id', as_index=True).sum(numeric_only=True)

print(df_aggregated)

                       lat        lon        area    p_mean   pet_mean  \
basin_id                                                                 
camels_01022500  44.607970 -67.935240  619.102595  3.203713  14.543954   
camels_01031500  45.175010 -69.314700  764.824149  3.289147  12.983978   
camels_01047000  44.869200 -69.955100  902.895076  3.275696  12.103014   
camels_01052500  44.877390 -71.057490  395.444910  3.401993  10.643175   
camels_01054200  44.390440 -70.979640  180.835783  3.664236  10.999606   
...                    ...        ...         ...       ...        ...   
lamah_47840      48.164517   9.522516  154.932286  3.024349   2.810292   
lamah_76175      48.398646   9.964808  484.426388  2.929337   4.001406   
lamah_76176      48.862387  10.351313  106.621005  2.530369   3.795519   
lamah_76184      48.299570   9.722481  167.318641  3.006783   3.648126   
lamah_76276      48.630146  10.154463  338.216741  2.754255   3.642578   

                  aridity  frac_snow 

In [24]:
df_aggregated
df_combined_info['basin_id'] = df_combined_info.index

In [25]:
df_combined_info[10:]

,timezone,name,country,lat,lon,area,p_mean,pet_mean,aridity,frac_snow,moisture_index,seasonality,high_prec_freq,high_prec_dur,low_prec_freq,low_prec_dur,basin_id
basin_id,,,,,,,,,,,,,,,,,
camels_01139000,America/New_York,"WELLS RIVER AT WELLS RIVER, VT",United States of America,44.150340,-72.065090,258.983212,3.142010,11.888000,3.783566,0.280448,-0.659372,0.555168,0.046547,1.109299,0.585461,2.882710,camels_01139000
camels_01144000,America/New_York,"WHITE RIVER AT WEST HARTFORD, VT",United States of America,43.714240,-72.418150,1777.408717,3.336646,12.160641,3.644570,0.281403,-0.650284,0.544912,0.046478,1.088141,0.576083,2.833670,camels_01144000
camels_01169000,America/New_York,"NORTH RIVER AT SHATTUCKVILLE, MA",United States of America,42.638420,-72.725090,233.453197,3.681383,13.664473,3.711777,0.305872,-0.677821,0.440245,0.054282,1.090784,0.610856,3.124650,camels_01169000
camels_01170100,America/New_York,"GREEN RIVER NEAR COLRAIN, MA",United States of America,42.703420,-72.670650,109.178943,3.581886,13.798289,3.852242,0.306876,-0.688454,0.429969,0.055240,1.093496,0.619344,3.176966,camels_01170100
camels_01181000,America/New_York,"WEST BRANCH WESTFIELD RIVER AT HUNTINGTON, MA",United States of America,42.237310,-72.895650,244.861975,3.765605,14.358677,3.813112,0.223039,-0.695800,0.370189,0.053802,1.100840,0.620234,3.200636,camels_01181000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
lamah_47840,Europe/Berlin,Kanzach at Unlingen,Germany,48.164517,9.522516,154.932286,3.024349,2.810292,0.929222,0.123289,0.164322,1.053874,0.032240,1.132212,0.526662,3.243676,lamah_47840
lamah_76175,Europe/Berlin,Blau at Ulm,Germany,48.398646,9.964808,484.426388,2.929337,4.001406,1.365976,0.134020,-0.116695,1.068867,0.032993,1.134118,0.525087,3.290862,lamah_76175
lamah_76176,Europe/Berlin,Eger at Bopfingen,Germany,48.862387,10.351313,106.621005,2.530369,3.795519,1.499986,0.079343,-0.143352,1.200170,0.034636,1.144796,0.555959,3.499354,lamah_76176


In [26]:
print(gdf)

             gauge_id                                           geometry
0     camels_01022500  POLYGON ((-67.97836 44.6131, -67.98141 44.6143...
1     camels_01031500  MULTIPOLYGON (((-69.31629 45.15325, -69.32144 ...
2     camels_01047000  POLYGON ((-70.10847 45.21669, -70.10609 45.213...
3     camels_01052500  POLYGON ((-71.10862 45.1273, -71.10402 45.1258...
4     camels_01054200  POLYGON ((-70.97999 44.39574, -70.97657 44.393...
...               ...                                                ...
6825      lamah_47840  POLYGON ((9.55616 48.16253, 9.56571 48.15944, ...
6826      lamah_76175  POLYGON ((9.8875 48.38333, 9.88529 48.38277, 9...
6827      lamah_76176  POLYGON ((10.38109 48.8752, 10.37946 48.87519,...
6828      lamah_76184  POLYGON ((9.75509 48.31759, 9.75364 48.31563, ...
6829      lamah_76276  POLYGON ((10.23328 48.68325, 10.23114 48.67988...

[6830 rows x 2 columns]


In [27]:
if 'gauge_id' in gdf.columns:
    gdf = gdf.set_index('gauge_id')

# Keep first occurrence per basin
# df_unique = df_combined_info[~df_combined_info.index.duplicated(keep='first')]


In [28]:
print(gdf)

                                                          geometry
gauge_id                                                          
camels_01022500  POLYGON ((-67.97836 44.6131, -67.98141 44.6143...
camels_01031500  MULTIPOLYGON (((-69.31629 45.15325, -69.32144 ...
camels_01047000  POLYGON ((-70.10847 45.21669, -70.10609 45.213...
camels_01052500  POLYGON ((-71.10862 45.1273, -71.10402 45.1258...
camels_01054200  POLYGON ((-70.97999 44.39574, -70.97657 44.393...
...                                                            ...
lamah_47840      POLYGON ((9.55616 48.16253, 9.56571 48.15944, ...
lamah_76175      POLYGON ((9.8875 48.38333, 9.88529 48.38277, 9...
lamah_76176      POLYGON ((10.38109 48.8752, 10.37946 48.87519,...
lamah_76184      POLYGON ((9.75509 48.31759, 9.75364 48.31563, ...
lamah_76276      POLYGON ((10.23328 48.68325, 10.23114 48.67988...

[6830 rows x 1 columns]


In [29]:
for val in df_combined_info.keys():
    gdf[val] = df_combined_info[val]

# for col in df_unique.columns:
#     gdf[col] = df_unique[col]

In [30]:
gdf[['timezone','name','country']]

,timezone,name,country
gauge_id,,,
camels_01022500,America/New_York,"Narraguagus River at Cherryfield, Maine",United States of America
camels_01031500,America/New_York,"Piscataquis River near Dover-Foxcroft, Maine",United States of America
camels_01047000,America/New_York,"Carrabassett River near North Anson, Maine",United States of America
camels_01052500,America/New_York,"Diamond River near Wentworth Location, NH",United States of America
camels_01054200,America/New_York,"Wild River at Gilead, Maine",United States of America
...,...,...,...
lamah_47840,Europe/Berlin,Kanzach at Unlingen,Germany
lamah_76175,Europe/Berlin,Blau at Ulm,Germany
lamah_76176,Europe/Berlin,Eger at Bopfingen,Germany


Turn it into a Map using folium

In [31]:
# control_bool = True
# caravan_map = folium.Map(
#                     # location        = [NL_outline.iloc[0].geometry.centroid.y, NL_outline.iloc[0].geometry.centroid.x],
#                     # zoom_start      = 8,
#                     # zoom_control    = control_bool,
#                     # scrollWheelZoom = control_bool,
#                     # dragging        = control_bool
#                     )

# catchments = gdf.to_crs(epsg=4326)

# folium.features.GeoJson(catchments,).add_to(caravan_map)

In [32]:
# # icon = folium.features.Icon(color="blue",icon="none")

# gjson = folium.features.GeoJson(
#                                 gdf,
#                                 # marker = folium.features.Marker(icon=icon)
#                                 ).add_to(caravan_map)

# # create
# popup = folium.features.GeoJsonPopup(
#     fields=['name','country','basin_id'],

# ).add_to(gjson)

In [33]:
# caravan_map.save(f"caravan_catchments_map.html")

In [52]:
LINK = "https://www.ewatercycle.org/seamless/main"

# 1. Add the done flag
gdf["done"] = gdf.index.isin(region_ids_done)

# 2. Add the link column as clickable HTML
gdf["link"] = gdf.apply(
    lambda row: f'<a href="{LINK}/{row.country}/{row.name}/step_4_analyse_executed.ipynb" target="_blank">Open Notebook</a>'
    if row.done else "Not yet implemented",
    axis=1
)

### Too big, split per contry: 

In [53]:
countries = gdf['country'].unique()

In [54]:
import warnings
warnings.simplefilter("ignore") # centroid gives : UserWarning: Geometry is in a geographic CRS

In [55]:
countries

array(['Mexico', 'United States of America', 'Brazil', 'Chile', 'Germany',
       'Canada', 'Austria', 'Switzerland', 'England', 'Scotland',
       'Australia', 'Czech Republic', 'Wales', 'Lichtenstein'],
      dtype=object)

In [56]:
# fix issue of overlapping polygons
gdf = gdf.sort_values("area",ascending=False)

In [57]:
for country in countries:
    gdf_country = gdf[gdf["country"] == country].copy()

    # Folium requires EPSG:4326
    gdf_country = gdf_country.to_crs(epsg=4326)

    # Map center
    country_map = folium.Map(
        location=[
            gdf_country.geometry.centroid.y.mean(),
            gdf_country.geometry.centroid.x.mean()
        ],
        zoom_start=6,
    )

    # Style: green if done, red if not
    def style_function(feature):
        done = feature["properties"]["done"]
        return {
            "fillColor": "green" if done else "red",
            "color": "black",
            "weight": 0.5,
            "fillOpacity": 0.6,
        }

    def highlight_function(feature):
        return {
            "fillColor": "yellow",
            "color": "black",
            "weight": 1.0,
            "fillOpacity": 0.7,
        }

    # Popup fields (including the new "link")
    popup_fields = [
        "basin_id",
        "name",
        "country",
        "area",
        "p_mean",
        "pet_mean",
        "aridity",
        "frac_snow",
        "moisture_index",
        "seasonality",
        "high_prec_freq",
        "high_prec_dur",
        "low_prec_freq",
        "low_prec_dur",
        "link",          # <-- add link here
    ]

    gjson = folium.GeoJson(
        gdf_country,
        style_function=style_function,
        highlight_function=highlight_function,
        tooltip=folium.GeoJsonTooltip(fields=["basin_id", "name"]),
        popup=folium.GeoJsonPopup(fields=popup_fields),
    ).add_to(country_map)

    # Save
    out_path = map_path / f"caravan_catchments_map_{country.replace(' ', '_')}.html"
    country_map.save(out_path)